# Census Data Imports (Work in Progress)

By Kenneth Burchfiel

Released under the MIT License

The US Census is a fantastic source of free demographic data. Thankfully, we can easily access large amounts of this data at once via Python, as demonstrated by this notebook. 

## Deciding where to move to start a family

Let's say that some NVCU graduating seniors are interested in settling down and raising a family a few years after they graduate. Because they'd prefer to live in a growing region rather than a declining one, they want to know which areas have seen the highest growth rates in recent years. They'd like to see this data both for all residents within each county *and* those aged 25-29.

In order to answer these questions, we'll use the Census API to retrieve US county population growth data for a selected set of years. We'll then use this data to calculate population growth rates across multiple periods.

## An introduction to the American Community Survey

Many Americans probably associate the US Census Bureau with its decennial Census. However, the Census Bureau also conducts the [American Community Survey](https://www.census.gov/data/developers/data-sets/acs-5year.html) each year, making it a great resource for recent demographic data.

This notebook will source data from the American Community Survey's 5-year estimates, which show an average of results for the past 5 years. (For example, the 2021 ACS5 dataset shows results between 2017 and 2021). The [1-year ACS estimates](https://www.census.gov/data/developers/data-sets/acs-1year.html) offer results for a more recent timeframe; however, because the 5-year estimates are sourced from a larger pool of data, they may be more reliable (especially for smaller regions). In addition, 1-year estimates aren't available for certain regions, such as zip codes.

For the sake of brevity, I'll often refer to the American Community Survey's 5-year estimates as the 'ACS5' survey.

# Introducing the Census API

In the process of working to answer the first question (regarding where to move to start a family), we'll also explore the Census API's capabilities and craft a Python function that will use it to efficiently retrieve data.

Importing relevant libraries and setting two configuration variables:

In [1]:
import time
program_start_time = time.time()
import pandas as pd
import numpy as np
from iteration_utilities import duplicates
pd.set_option('display.max_columns', 1000)
import lxml # Necessary for reading online HTML tables into Pandas

render_for_pdf = False
if render_for_pdf == True:
    pd.set_option('display.max_columns', 4)


acs5_year = 2021 # By updating this variable when future American 
# Community Surveys get released, you should be able to retrieve the most
# recent data possible. (If changes to the survey's format are made,
# however, updates to the scripts may be necessary.)

# Note: I had originally set acs5_year to 2022, the latest year for which
# ACS5 data were available at the time. However, due to a recent change
# in Connecticut's county-equivalent boundaries (see
# https://www.federalregister.gov/documents/2022/06/06/2022-12063/
# change-to-county-equivalents-in-the-state-of-connecticut for more
# information), ACS5 population growth data between previous
# years and 2022 appeared to be unavailable for that state. Therefore,
# I chose to retrieve data for 2021 instead. 

download_variable_list = True # If set to True, a new list of variables
# will be downloaded from the Census API website. If False, this list of 
# variables will instead be read in from a local .csv copy (thus saving
# processing time).

## Importing a Census API Key

You can obtain a free Census API key [at this website](https://api.census.gov/data/key_signup.html). The following cell imports my own personal key, so you'll need to replace this code with one that loads in your own API key.

In [2]:
with open ('census_api_key_path.txt') as file:
    key_path = file.read()
with open(key_path) as file:
    key = file.read()

The Census offers detailed API documentation that makes retrieving data from it relatively straightforward. For instance, you'll probably find the [Census Data API User Guide](https://www.census.gov/content/dam/Census/data/developers/api-user-guide/api-guide.pdf) to be helpful in applying the Census API.

[This list](https://api.census.gov/data/2021/acs/acs5/examples.html) of ACS5 API call examples is another great resource. One of the sample URLs shown on this page for retrieving county-level data appears as follows:

https://api.census.gov/data/2021/acs/acs5?get=NAME,B01001_001E&for=county:*&key=YOUR_KEY_GOES_HERE

If you replace the 'YOUR_KEY_GOES_HERE' component of the URL with your actual key, then enter this link into your web browser, you'll receive a very long list of counties, population values, and state and county codes. The top of the list for the 2021 ACS5 looks like this:

```
[["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58239","01","001"],
["Baldwin County, Alabama","227131","01","003"],
["Barbour County, Alabama","25259","01","005"],
["Bibb County, Alabama","22412","01","007"],
["Blount County, Alabama","58884","01","009"],
["Bullock County, Alabama","10386","01","011"],
```

'B01001_001E' refers to the total population estimates for a given county. We can find this out by going to the [2021 ACS5's Detailed Tables page for the](https://api.census.gov/data/2021/acs/acs5/variables.html) and navigating to the row with a 'Name' value of 'B01001_001E'. This link, which may take a little while to fully load, is available on the [ACS5 API Documentation Page](https://www.census.gov/data/developers/data-sets/acs-5year.html).


We can use `pd.read_json()` to easily read this same data into a DataFrame:

In [3]:
df_results = pd.read_json(
    f'https://api.census.gov/data/{acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# read_json documentation:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/
# pandas.read_json.html
df_results.head()

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007


At this point, the DataFrame's columns are [0, 1, 2, 3], whereas the columns we want to use are stored within the first row. The following code sets these row values as our column values, then deletes this row:

In [4]:
df_results.columns = df_results.iloc[0]
df_results.drop(0, inplace = True)

df_results.head()

,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007
5,"Blount County, Alabama",58884,01,009


## Retrieving our Data

In order to determine which variable codes to enter into our script, we'll first need to review a list of all American Community Survey variables and the overall groups into which they fit. This list is available on the Census website ([here's the copy for 2021](https://api.census.gov/data/2021/acs/acs5/variables.html)), but we can also use Pandas to import them into a DataFrame, as shown below.

In [5]:
if download_variable_list == True:
    df_variables_page = pd.read_html(
        f'https://api.census.gov/data/{acs5_year}/acs/acs5/variables.html')[0] 
    # [0] selects the first HTML table found on this page.
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.read_html.html
    # for more information on pd.read_html().
        
    # Some rows in this table contain items other than demographic 
    # variables (e.g. region names). We can exclude them by selecting 
    # only rows that begin with 'Estimate'. (Another option would have 
    # been to filter out rows with N/A 'Group' entries (i.e. 
    # df_variables.query("Group.isna() == False")), 
    # but this would have left a couple non-variable rows in place.
    
    df_variables = df_variables_page[
    df_variables_page['Label'].str[0:8] == 'Estimate'].copy(
    ).reset_index(drop=True)
    # Saving this table to a local .csv file:
    df_variables.to_csv(f'Datasets/{acs5_year}_variables.csv', 
    index = False)
else: # Reading a local copy of this dataset instead, which should 
    # take much less time. 
    df_variables = pd.read_csv(
        f'Datasets/{acs5_year}_variables.csv')
df_variables.head()

,Name,Label,Concept,Required,Attributes,Limit,Predicate Type,Group,Unnamed: 8
0,B01001_001E,Estimate!!Total:,SEX BY AGE,not required,"B01001_001EA, B01001_001M, B01001_001MA",0,int,B01001,NaN
1,B01001_002E,Estimate!!Total:!!Male:,SEX BY AGE,not required,"B01001_002EA, B01001_002M, B01001_002MA",0,int,B01001,NaN
2,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,SEX BY AGE,not required,"B01001_003EA, B01001_003M, B01001_003MA",0,int,B01001,NaN
3,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,SEX BY AGE,not required,"B01001_004EA, B01001_004M, B01001_004MA",0,int,B01001,NaN
4,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,SEX BY AGE,not required,"B01001_005EA, B01001_005M, B01001_005MA",0,int,B01001,NaN


With over 28,000 individual variables, it could take a very long time to identify the items you'd like to retrieve from the Census. We can make this search process somewhat easier by creating a separate *groups* table that shows only unique group names and their written descriptions (e.g. 'Sex by Age').

In [6]:
df_groups = df_variables.drop_duplicates(
    'Group')[['Concept', 'Group']].copy(
    ).reset_index(drop=True)
df_groups.head()

,Concept,Group
0,SEX BY AGE,B01001
1,SEX BY AGE (WHITE ALONE),B01001A
2,SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE),B01001B
3,SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ...,B01001C
4,SEX BY AGE (ASIAN ALONE),B01001D


We'll save this group table to a local .csv file as well:

In [7]:
df_groups.to_csv(f'Datasets/{acs5_year}_groups.csv', 
                 index = False)

In order to find variables of interest, I recommend first searching for keywords of interest within the group table (which is much smaller in size) in order to identify relevant group IDs. Next, you can search for those group IDs inside the variables table in order to find the exact metrics to request from the Census API.

The following table stores variables for three separate groups: (1) the total population; (2) all males aged 25 to 29 years; and (3) all females aged 25 to 29 years. (The B01001 table that stores these values didn't have an entry for all people aged 25 to 29; we'll get around this limitation by retrieving sex-specific population totals within this age group, then adding those totals together.)

In [8]:
grad_destinations_variable_list = [
    'B01001_001E', 'B01001_011E',
    'B01001_035E']

The demographic columns in the Census API's output are labeled with their variable names (e.g. 'B01001_001E'). These names are concise, but you'll need a copy of the original variable list to interpret them. Therefore, I chose to replace these column names with a combination of the 'Label', 'Concept', and 'Name' entries in the original variable list. These column names are very long, but they do make the output easier to interpret (while also preserving the original names for reference). 

In addition, if the description corresponding to a variable name happens to change from one year to another, the use of aliases will help you identify that change. (This will help prevent you from treating two different data types that happened to use the same variable code in different years as equal. However, I would imagine that the Census wouldn't repurpose variable codes in this way.)

The following function assists with this replacement by creating a dictionary whose keys are the original field names (e.g. 'B0101_001E') and whose values are the replacement names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').

In [9]:
def create_variable_aliases(df_variables, variable_list):
    '''This function creates a dictionary whose keys are 
    the original 'Name' values (e.g. 'B001_001E') within a variable
    list on the Census API website and whose values are the replacement 
    names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').
    This resulting dictionary can then be passed to a df.rename() call
    within retrieve_census_data() in order to make the output of that
    function easier to interpret.
    
    df_variables: A DataFrame containing a list of Census variables. For
    an example of this list for the 2021 American Community Survey (5-Year 
    Estimates), visit: 
    https://api.census.gov/data/2021/acs/acs5/examples.html .
    
    variable_list: The list of variables to rename 
    (e.g. ['B01001_001E', 'B01001_002E']).
    '''
    # Creating a DataFrame that contains the information needed for the
    # updated column names:
    df_aliases = df_variables.query(
        "Name in @variable_list")[['Name', 'Label', 'Concept']].copy()
    # Creating a new 'Description' column that will replace the original
    # output field names:
    df_aliases['Description'] = (df_aliases['Concept'] 
                                 + '_' + df_aliases['Label'] 
                                 + ' (' + df_aliases['Name'] + ')')
    # Creating a dictionary whose keys are the original field names and 
    # whose values are the new 'Description' entries that were 
    # just created:
    alias_dict = df_aliases.set_index('Name').to_dict()['Description']
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.DataFrame.to_dict.html
    return alias_dict

Creating our aliases:

In [10]:
grad_destinations_alias_dict = create_variable_aliases(
    df_variables = df_variables, 
    variable_list = grad_destinations_variable_list)
grad_destinations_alias_dict

{'B01001_001E': 'SEX BY AGE_Estimate!!Total: (B01001_001E)',
 'B01001_011E': 'SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
 'B01001_035E': 'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'}

## Defining a Census data retrieval function

The following function simplifies the process of retrieving data from the Census API. It also enables the user to rename variable fields (e.g. 'B01001_001E') with aliases for those fields (e.g. 'Sex by Age_Estimate!!Total: (B01001_001E)'), but this option is disabled by default. In addition, it allows more than 50 variables to be retrieved at the same time, thus making it easier to retrieve especially large datasets.

[Note: currently, this function only supports data retrieval for the ACS 5-year and 1-year estimates. However, I may add in the ability to retrieve decennial Census data in the future.]

In [11]:
def retrieve_census_data(survey, year, region, key, variable_list,
                         rename_data_fields = False, 
                         field_names_dict = {}):
    '''This function (which I plan to expand) retrieves data from the US
    Census API. It accommodates more than 50 variables.
    
    survey: the survey from which to retrieve data. The only arguments
    currently supported are 'acs5' and 'acs1' (for the American Community 
    Survey 5-Year and 1-Year estimates, respectively).
    
    year: the year for which you wish to retrieve survey data. Note that,
    When region is set to 'acs5', the survey results will include data
    for the 5 years leading up to (and including) the 'year' argument.
    (For example, if you set 'year' to 2021, you'll retrieve ACS5 data
    from 2017 to 2021 (inclusive).)
    
    
    region: The geographic level at which you wish to retrieve data. 
    Examples include 'us', 'state', 'county', 'zip', 'msa' 
    (for metropolitan/micropolitan statistical area data), and 'csa' 
    (for combined statistical area data); 
    however, other regions are supported as well. Consult your survey's 
    API examples page for other options. (For instance, if you wanted to 
    retrieve data by urban area within the 2021 ACS5, you could go to 
    https://api.census.gov/data/2021/acs/acs5/examples.html, then search
    for 'urban area.' The Urban Area URL ends with
    '&for=urban%20area:*&key=YOUR_KEY_GOES_HERE'. Therefore, you'd want to
    use 'urban%20area' as your 'region' argument.)   

    (Note: 'zip' will retrieve results by Zip Code
    Tabulation Area, which are similar to (but not identical to)
    # zip codes. See 
    # https://en.wikipedia.org/wiki/ZIP_Code_Tabulation_Area
    # for more information.
    
    variable_list: The list of variables for which to retrieve data.

    key: your personal Census API key.

    rename_data_fields: set to True to replace column names in your 
    dataset with new entries of your choice.

    field_names_dict: A dictionary that stores the original variable names
    retrieved by the Census (e.g. 'B01001_001E' as keys and your desired
    replacements as values. Example: 
    {'B01001_001E': 'Sex by Age_Estimate!!Total:_B01001_001E',
     'B01001_002E': 'Sex by Age_Estimate!!Total:!!Male:_B01001_002E'}'
     
    '''

    # Using the iteration_utilities library to check for duplicate
    # values within variable_list (which could cause issues later on):
    # The following code is based on
    # https://iteration-utilities.readthedocs.io/en/latest/generated/
    # duplicates.html
    duplicate_variables = list(duplicates(variable_list))
    
    if len(duplicate_variables) > 0:
        raise ValueError(f"The following variables appear more than once \
in your variable list: {duplicate_variables}")
    
    if survey == 'acs5':
        survey_string = 'acs/acs5'

    elif survey == 'acs1':
        survey_string = 'acs/acs1'
    
    else:
        raise ValueError("This survey type is not currently supported by \
                         the function.")

    
    # Converting simplified region names into strings that the API 
    # function will recognize:
    if region == 'zip':
        region = 'zip%20code%20tabulation%20area' # Based on
        # the ZCTA example within
        # https://api.census.gov/data/2021/acs/acs5/examples.html
    
    if region == 'csa':
        region = 'combined%20statistical%20area'
    
    if region == 'msa':
        region = 'metropolitan%20statistical\
%20area/micropolitan%20statistical%20area'

    
    # Only 50 variables can be retrieved from the Census API at a time 
    # using the approach shown in this function. The following code 
    # accommodates this limitation by splitting variable_list into 
    # sublists of up to 49 variables. The data retrieved for the variables 
    # in these sublists will then get merged back together.
    # (49 variables are retrieved at a time instead of 50 because it 
    # appears that the initial 'NAME' variable also counts towards 
    # the 50-variable limit.)
    
    i = 0
       
    while i < len(variable_list): # i.e. while there
        # are still more variables to iterate through
        variable_sublist = variable_list[i:i+49] # This line reads the 
        # next 49 variables from variable_list into a sublist that can 
        # then be\ passed to the API
        # print("variable_sublist:", variable_sublist)
        # Converting the list of variables into a string that can be 
        # passed to the API call:
        # (The Census API guide at
        # https://www.census.gov/content/dam/Census/data/developers/
        # api-user-guide/api-guide.pdf
        # demonstrates how to call multiple census variables at once.)
        variable_string = ','.join(variable_sublist)
        # print("variable_string:",variable_string)
    
        # Retrieving data via the Census API:
        # This line was originally based on an example found in
        # https://api.census.gov/data/2022/acs/acs5/examples.html .
    
        # read_json documentation:
        # https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.read_json.html

        api_url = f'https://api.census.gov/data/{year}/\
{survey_string}?get=NAME,{variable_string}&for={region}:*&key={key}'
        # print(api_url)
        
        df_results = pd.read_json(api_url)
    
        # At this point, the DataFrame's columns are a list of integers; 
        # the desired column names are stored within the first row. 
        # The following code resolves this issue by setting these row 
        # values as the column values and then deleting this row.
    
        df_results.columns = df_results.iloc[0]
        df_results.drop(0, inplace = True)


        # Determining which merge keys to use when combining API results
        # for different sublists together:
        # This is made more complicated by the fact that results for 
        # different regions will have different identifier
        # columns (e.g. 'NAME', 'county', and 'state' for county data but 
        # only 'NAME' and 'state' for state data). However, we can 
        # accommodate this behavior by simply initializing our list of 
        # merge keys as the set of all columns that are *not* also 
        # variable columns.
        if i == 0: # This step only needs to be performed for our first
            # sublist of variables, since merge keys for other sublists
            # will be identical.
            merge_keys = list(set(df_results.columns) 
              - set(variable_sublist))
            # print("merge_keys:",merge_keys)

        if i == 0: # Since this is the first set 
            # of results, we can initialize df_combined_results 
            # as a copy of df_results.
            df_combined_results = df_results.copy()
        else: # Merging our latest set of results into df_results:
            df_combined_results = df_combined_results.merge(
                df_results, on = merge_keys,
                how = 'outer').copy()
            # Added .copy() here in response to a data fragmentation 
        # warning

        i += 49 
        # Allows the function to iterate through the next 49 variables
        # within variable_list

        
    # Converting variable columns to numeric data types:
    for column in variable_list:
        # print(f"Now converting {column} to a numeric type.")
        df_combined_results[column] = pd.to_numeric(
            df_combined_results[column])
        # pd.to_numeric() allows for either integer or float outputs
        # depending on the nature of the original data.
        # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.to_numeric.html

    # Replacing column names with aliases if requested:
    if rename_data_fields == True:
        df_combined_results.rename(
            columns = field_names_dict, inplace = True)

    # The following for loop moves all of the merge keys (e.g. geographic
    # identifiers) to the left side of the table. This is particularly
    # useful when retrieving longer lists of variables, as otherwise,
    # certain keys can get buried in the middle of the dataset
    for i in range(len(merge_keys)):
        df_combined_results.insert(
            i, merge_keys[i], 
            df_combined_results.pop(merge_keys[i]))

    # Adding a 'Year' column to the left of all existing DataFrame columns:
    # (this will prove particularly
    # helpful when comparing data from different years.)
    df_combined_results.insert(0, 'Year', year)
    
    return df_combined_results

(The following code allowed me to test out retrieve_census_data for a particularly long variable list.)

In [12]:
# test_list = list(df_variables['Name'][0:151])

# test_alias_dict = create_variable_aliases(
#     df_variables = df_variables, 
#     variable_list = test_list)

# test_acs5_data = retrieve_census_data(
#     survey = 'acs5', year = acs5_year, region = 'county',
#     variable_list = test_list, 
#     rename_data_fields = True, 
#     field_names_dict = test_alias_dict, key = key)

# test_acs5_data

Next, we'll define a list of years for which we would like to retrieve Census data. In order to make this code easier to use in future years, I'll define these years as an offset of acs5_year rather than hardcoding them.

In [13]:
years_to_retrieve = [acs5_year - 12, acs5_year -10, 
                     acs5_year - 8,
                     acs5_year - 6,
                     acs5_year - 5,
                     acs5_year]
# American Community Survey 1-year estimates weren't available in 2020,
# so you'll want to remove that year from your list if it happens to be present.
# However, because I decided to use 5-year rather than 1-year estimates,
# I commented out this line.
# if 2020 in years_to_retrieve:
#     years_to_retrieve.remove(2020)
years_to_retrieve

[2009, 2011, 2013, 2015, 2016, 2021]

At this point, it's a good idea to confirm that our three variable codes ('B01001_001E', 'B01001_011E', and 'B01001_035E') had the same meaning for all the years whose data we'll be retrieving. We can do so by running the following code, which retrieves these variables and their corresponding descriptions for all of the years in years_to_retrieve.

(Due to the size of the variables.html page, this code can take a while to run, so I commented it out below.)

In [14]:
# var_meanings_by_year_df_list = []
# for year in years_to_retrieve:
#     df_var_list = pd.read_html(
#         f'https://api.census.gov/data/{year}/acs/acs5/variables.html')[
#     0][['Name', 'Label', 'Concept']].query(
#         "Name in @grad_destinations_variable_list")
#     df_var_list.insert(0, 'Year', year)
#     var_meanings_by_year_df_list.append(df_var_list)
# df_var_meanings_by_year = pd.concat([df for df in var_meanings_by_year_df_list])
# df_var_meanings_by_year.to_csv('var_meanings_by_year.csv', index = False)
# df_var_meanings_by_year

Here's a saved .csv copy of this table that confirms that these codes had the same meaning in each of the years we're analyzing:

In [15]:
pd.read_csv('var_meanings_by_year.csv')

,Year,Name,Label,Concept
0,2010,B01001_001E,Estimate!!Total,SEX BY AGE
1,2010,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
2,2010,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
3,2012,B01001_001E,Estimate!!Total,SEX BY AGE
4,2012,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
5,2012,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
6,2014,B01001_001E,Estimate!!Total,SEX BY AGE
7,2014,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
8,2014,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
9,2016,B01001_001E,Estimate!!Total,SEX BY AGE


We're now ready to retrieve our population totals for the years referenced in years_to_retrieve. We'll do so by calling retrieve_census_data() for each of these years via a for loop, then adding their respective DataFrames together using pd.concat().

(Note: Only 5825 results showed up when I requested ACS1 data (as opposed to over 25,000 results for ACS5 data), so many counties were not getting incorporated within the ACS1 results. Therefore, the ACS5 should be the best survey to use for evaluating county-level growth.)

In [16]:
census_data_by_year_df_list = []
for year in years_to_retrieve:
    df_data = retrieve_census_data(
        survey = 'acs5', year = year, 
        region = 'county',
        variable_list = grad_destinations_variable_list, 
        rename_data_fields = True, field_names_dict = grad_destinations_alias_dict,
        key = key)
    census_data_by_year_df_list.append(df_data)
df_growth_data_by_year = pd.concat(df for df in census_data_by_year_df_list)
# Removing Puerto Rico from our list of results so as to focus only on counties
# and county equivalents within the 50 US states + DC:
df_growth_data_by_year.query("state != '72'", inplace = True)
df_growth_data_by_year

,Year,state,county,NAME,SEX BY AGE_Estimate!!Total: (B01001_001E),SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E),SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)
1,2009,13,091,"Dodge County, Georgia",19695,634,574
2,2009,13,093,"Dooly County, Georgia",11641,531,352
3,2009,13,095,"Dougherty County, Georgia",95330,3264,3900
4,2009,13,097,"Douglas County, Georgia",122657,4066,4353
5,2009,13,099,"Early County, Georgia",11812,275,282
...,...,...,...,...,...,...,...
3139,2021,56,037,"Sweetwater County, Wyoming",42459,1320,1259
3140,2021,56,039,"Teton County, Wyoming",23319,1142,905
3141,2021,56,041,"Uinta County, Wyoming",20514,574,567
3142,2021,56,043,"Washakie County, Wyoming",7768,175,200


The following cell adds together male and female population totals in order to calculate the total number of 25- to 29-year-olds within each county. It also simplifies the original total population column name in order to improve its readability.

In [17]:
df_growth_data_by_year['Total_Pop_25_to_29'] = (df_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)'] + 
df_growth_data_by_year[
'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'])
df_growth_data_by_year.rename(
    columns = {'SEX BY AGE_Estimate!!Total: (B01001_001E)':'Total_Pop'}, 
inplace = True)
df_growth_data_by_year.drop(
    ['SEX BY AGE_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
     'SEX BY AGE_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'],
     axis = 1, inplace = True)
                             

df_growth_data_by_year

,Year,state,county,NAME,Total_Pop,Total_Pop_25_to_29
1,2009,13,091,"Dodge County, Georgia",19695,1208
2,2009,13,093,"Dooly County, Georgia",11641,883
3,2009,13,095,"Dougherty County, Georgia",95330,7164
4,2009,13,097,"Douglas County, Georgia",122657,8419
5,2009,13,099,"Early County, Georgia",11812,557
...,...,...,...,...,...,...
3139,2021,56,037,"Sweetwater County, Wyoming",42459,2579
3140,2021,56,039,"Teton County, Wyoming",23319,2047
3141,2021,56,041,"Uinta County, Wyoming",20514,1141
3142,2021,56,043,"Washakie County, Wyoming",7768,375


The following code applies pd.pivot() to place all population totals for a given county within the same row, thus making future growth calculations easier.

In [18]:
df_growth_data_by_year_for_pivot = df_growth_data_by_year.copy()
df_growth_data_by_year_pivot = df_growth_data_by_year_for_pivot.pivot(
    columns = 'Year', index = ['NAME', 'county', 'state'],
# The values could be named explicitly, but since pivot() will infer them
    # automatically, there's no need to do so. The following code is thus
    # commented out.
    # values = ['Total_Pop',
#           'Total_Pop_25_to_29']
).reset_index()
df_growth_data_by_year_pivot.columns = df_growth_data_by_year_pivot.columns.to_flat_index()

df_growth_data_by_year_pivot.head()

,"(NAME, )","(county, )","(state, )","(Total_Pop, 2009)","(Total_Pop, 2011)","(Total_Pop, 2013)","(Total_Pop, 2015)","(Total_Pop, 2016)","(Total_Pop, 2021)","(Total_Pop_25_to_29, 2009)","(Total_Pop_25_to_29, 2011)","(Total_Pop_25_to_29, 2013)","(Total_Pop_25_to_29, 2015)","(Total_Pop_25_to_29, 2016)","(Total_Pop_25_to_29, 2021)"
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0


In [19]:
df_growth_data_by_year_pivot.columns = [
    column[0] + '_' + str(column[1]) if isinstance(column[1], int) 
    else column[0] for column in df_growth_data_by_year_pivot.columns]
df_growth_data_by_year_pivot

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,"Yuma County, Arizona",027,04,188983.0,193995.0,199026.0,202987.0,203292.0,202944.0,10004.0,12927.0,13539.0,14140.0,14422.0,14904.0
3146,"Yuma County, Colorado",125,08,9630.0,9960.0,10093.0,10185.0,10150.0,9944.0,376.0,566.0,592.0,681.0,614.0,626.0
3147,"Zapata County, Texas",505,48,13561.0,13853.0,14164.0,14308.0,14335.0,13945.0,896.0,900.0,962.0,943.0,936.0,876.0
3148,"Zavala County, Texas",507,48,11620.0,11700.0,11849.0,12060.0,12107.0,9900.0,691.0,712.0,726.0,780.0,784.0,698.0


The following code was derived from my create_enrollment_comparisons() function within  catholic_school_enrollment_trends_v4.ipynb (available at https://github.com/kburchfiel/catholic_school_enrollment/blob/main/catholic_school_enrollment_trends_v4.ipynb).

In [20]:
def create_comparison_fields(df, field_name, year_list,
                             field_year_separator = '_'):
    latest_year = year_list[-1]
    for year in year_list[:-1]:

        # Calculating the nominal change between the two years:
        df[f'{year}-{latest_year} {field_name} Change'] = (
        df[f'{field_name}{field_year_separator}{latest_year}'] 
        - df[f'{field_name}{field_year_separator}{year}'] )

        # Calculating the percentage change:
        df[f'{year}-{latest_year} {field_name} % Change'] = 100*((
        df[f'{field_name}{field_year_separator}{latest_year}'] 
        / df[f'{field_name}{field_year_separator}{year}']) - 1)

    
        # Calculating ranks and percentiles for both the nominal 
        # change and % change columns:
        df[f'{year}-{latest_year} Change Rank'] = df[
        f'{year}-{latest_year} {field_name} Change'].rank(
            ascending=False, method = 'min')
        
        df[f'{year}-{latest_year} Change Percentile'] = 100 * df[
        f'{year}-{latest_year} {field_name} Change'].rank(
            pct=True, ascending=True, method='max')
        
        df[f'{year}-{latest_year} % Change Rank'] = (
            df[f'{year}-{latest_year} {field_name} % Change'].rank(
            ascending=False, method = 'min'))
        
        df[f'{year}-{latest_year} % Change Percentile'] = (
            100 * df[
            f'{year}-{latest_year} {field_name} % Change'].rank(
            pct=True, ascending=True, method='max'))

In [21]:
create_comparison_fields(
    df = df_growth_data_by_year_pivot,
    field_name = 'Total_Pop',
    year_list = years_to_retrieve,
    field_year_separator='_')
                         

In [22]:
df_growth_data_by_year_pivot.head(5)

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021,2009-2021 Total_Pop Change,2009-2021 Total_Pop % Change,2009-2021 Change Rank,2009-2021 Change Percentile,2009-2021 % Change Rank,2009-2021 % Change Percentile,2011-2021 Total_Pop Change,2011-2021 Total_Pop % Change,2011-2021 Change Rank,2011-2021 Change Percentile,2011-2021 % Change Rank,2011-2021 % Change Percentile,2013-2021 Total_Pop Change,2013-2021 Total_Pop % Change,2013-2021 Change Rank,2013-2021 Change Percentile,2013-2021 % Change Rank,2013-2021 % Change Percentile,2015-2021 Total_Pop Change,2015-2021 Total_Pop % Change,2015-2021 Change Rank,2015-2021 Change Percentile,2015-2021 % Change Rank,2015-2021 % Change Percentile,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Change Rank,2016-2021 Change Percentile,2016-2021 % Change Rank,2016-2021 % Change Percentile
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0,-973.0,-3.838719,2627.0,16.262755,2367.0,24.553571,-1141.0,-4.471879,2625.0,16.353204,2309.0,26.426522,-859.0,-3.404272,2521.0,19.694073,2204.0,29.796048,-623.0,-2.492299,2451.0,21.999363,2138.0,31.964343,-577.0,-2.312533,2482.0,21.012416,2174.0,30.818211
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0,-1416.0,-2.375201,2789.0,11.096939,2158.0,31.218112,-3230.0,-5.258017,3018.0,3.825311,2451.0,21.899904,-3647.0,-5.896810,3049.0,2.868069,2615.0,16.698534,-3963.0,-6.375175,3083.0,1.878383,2760.0,12.161732,-4172.0,-6.688899,3102.0,1.273480,2837.0,9.710283
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0,-5134.0,-13.327449,3091.0,1.466837,3032.0,3.348214,-313.0,-0.928756,1968.0,37.296780,1700.0,45.839974,99.0,0.297396,1385.0,55.895475,1457.0,53.601020,273.0,0.824400,1244.0,60.426616,1335.0,57.529449,328.0,0.992136,1194.0,62.018465,1321.0,57.975167
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0,116455.0,31.577506,53.0,98.341837,81.0,97.448980,97072.0,25.007342,51.0,98.406120,67.0,97.896079,83573.0,20.806228,49.0,98.470363,62.0,98.056087,67745.0,16.226308,40.0,98.758357,69.0,97.835084,59448.0,13.961550,40.0,98.758357,68.0,97.866921
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0,-116.0,-1.535407,1927.0,38.584184,2023.0,35.522959,-260.0,-3.377062,1884.0,39.974498,2119.0,32.483264,-149.0,-1.963627,1734.0,44.773741,1931.0,38.495857,13.0,0.175061,1513.0,51.862464,1493.0,52.499204,109.0,1.487040,1409.0,55.173512,1202.0,61.763770


In [23]:
df_growth_data_by_year_pivot

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021,2009-2021 Total_Pop Change,2009-2021 Total_Pop % Change,2009-2021 Change Rank,2009-2021 Change Percentile,2009-2021 % Change Rank,2009-2021 % Change Percentile,2011-2021 Total_Pop Change,2011-2021 Total_Pop % Change,2011-2021 Change Rank,2011-2021 Change Percentile,2011-2021 % Change Rank,2011-2021 % Change Percentile,2013-2021 Total_Pop Change,2013-2021 Total_Pop % Change,2013-2021 Change Rank,2013-2021 Change Percentile,2013-2021 % Change Rank,2013-2021 % Change Percentile,2015-2021 Total_Pop Change,2015-2021 Total_Pop % Change,2015-2021 Change Rank,2015-2021 Change Percentile,2015-2021 % Change Rank,2015-2021 % Change Percentile,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Change Rank,2016-2021 Change Percentile,2016-2021 % Change Rank,2016-2021 % Change Percentile
0,"Abbeville County, South Carolina",001,45,25347.0,25515.0,25233.0,24997.0,24951.0,24374.0,1463.0,1239.0,1118.0,1191.0,1226.0,1292.0,-973.0,-3.838719,2627.0,16.262755,2367.0,24.553571,-1141.0,-4.471879,2625.0,16.353204,2309.0,26.426522,-859.0,-3.404272,2521.0,19.694073,2204.0,29.796048,-623.0,-2.492299,2451.0,21.999363,2138.0,31.964343,-577.0,-2.312533,2482.0,21.012416,2174.0,30.818211
1,"Acadia Parish, Louisiana",001,22,59616.0,61430.0,61847.0,62163.0,62372.0,58200.0,4123.0,4217.0,4172.0,4126.0,4240.0,3705.0,-1416.0,-2.375201,2789.0,11.096939,2158.0,31.218112,-3230.0,-5.258017,3018.0,3.825311,2451.0,21.899904,-3647.0,-5.896810,3049.0,2.868069,2615.0,16.698534,-3963.0,-6.375175,3083.0,1.878383,2760.0,12.161732,-4172.0,-6.688899,3102.0,1.273480,2837.0,9.710283
2,"Accomack County, Virginia",001,51,38522.0,33701.0,33289.0,33115.0,33060.0,33388.0,2469.0,1842.0,1810.0,1669.0,1819.0,1907.0,-5134.0,-13.327449,3091.0,1.466837,3032.0,3.348214,-313.0,-0.928756,1968.0,37.296780,1700.0,45.839974,99.0,0.297396,1385.0,55.895475,1457.0,53.601020,273.0,0.824400,1244.0,60.426616,1335.0,57.529449,328.0,0.992136,1194.0,62.018465,1321.0,57.975167
3,"Ada County, Idaho",001,16,368791.0,388174.0,401673.0,417501.0,425798.0,485246.0,32200.0,29327.0,29494.0,30142.0,30646.0,33625.0,116455.0,31.577506,53.0,98.341837,81.0,97.448980,97072.0,25.007342,51.0,98.406120,67.0,97.896079,83573.0,20.806228,49.0,98.470363,62.0,98.056087,67745.0,16.226308,40.0,98.758357,69.0,97.835084,59448.0,13.961550,40.0,98.758357,68.0,97.866921
4,"Adair County, Iowa",001,19,7555.0,7699.0,7588.0,7426.0,7330.0,7439.0,284.0,425.0,392.0,362.0,362.0,411.0,-116.0,-1.535407,1927.0,38.584184,2023.0,35.522959,-260.0,-3.377062,1884.0,39.974498,2119.0,32.483264,-149.0,-1.963627,1734.0,44.773741,1931.0,38.495857,13.0,0.175061,1513.0,51.862464,1493.0,52.499204,109.0,1.487040,1409.0,55.173512,1202.0,61.763770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,"Yuma County, Arizona",027,04,188983.0,193995.0,199026.0,202987.0,203292.0,202944.0,10004.0,12927.0,13539.0,14140.0,14422.0,14904.0,13961.0,7.387437,415.0,86.798469,866.0,72.417092,8949.0,4.613005,461.0,85.336309,892.0,71.597067,3918.0,1.968587,579.0,81.580625,1161.0,63.033779,-43.0,-0.021184,1606.0,48.901624,1531.0,51.289398,-348.0,-0.171182,2244.0,28.589621,1624.0,48.328558
3146,"Yuma County, Colorado",125,08,9630.0,9960.0,10093.0,10185.0,10150.0,9944.0,376.0,566.0,592.0,681.0,614.0,626.0,314.0,3.260644,1504.0,52.072704,1324.0,57.812500,-16.0,-0.160643,1555.0,50.462225,1562.0,50.239082,-149.0,-1.476271,1734.0,44.773741,1824.0,41.905672,-241.0,-2.366225,1994.0,36.548870,2110.0,32.855778,-206.0,-2.029557,2019.0,35.752945,2104.0,33.046800
3147,"Zapata County, Texas",505,48,13561.0,13853.0,14164.0,14308.0,14335.0,13945.0,896.0,900.0,962.0,943.0,936.0,876.0,384.0,2.831650,

In [24]:
range_for_sorting = 5
range_start_year = acs5_year - range_for_sorting 
sort_column = f'{range_start_year}-{acs5_year} \
% Change Rank'


df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 100000").sort_values(
    sort_column).dropna(subset = sort_column)

,NAME,county,state,Total_Pop_2009,Total_Pop_2011,Total_Pop_2013,Total_Pop_2015,Total_Pop_2016,Total_Pop_2021,Total_Pop_25_to_29_2009,Total_Pop_25_to_29_2011,Total_Pop_25_to_29_2013,Total_Pop_25_to_29_2015,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2021,2009-2021 Total_Pop Change,2009-2021 Total_Pop % Change,2009-2021 Change Rank,2009-2021 Change Percentile,2009-2021 % Change Rank,2009-2021 % Change Percentile,2011-2021 Total_Pop Change,2011-2021 Total_Pop % Change,2011-2021 Change Rank,2011-2021 Change Percentile,2011-2021 % Change Rank,2011-2021 % Change Percentile,2013-2021 Total_Pop Change,2013-2021 Total_Pop % Change,2013-2021 Change Rank,2013-2021 Change Percentile,2013-2021 % Change Rank,2013-2021 % Change Percentile,2015-2021 Total_Pop Change,2015-2021 Total_Pop % Change,2015-2021 Change Rank,2015-2021 Change Percentile,2015-2021 % Change Rank,2015-2021 % Change Percentile,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Change Rank,2016-2021 Change Percentile,2016-2021 % Change Rank,2016-2021 % Change Percentile
1234,"Hays County, Texas",209,48,141371.0,152827.0,164144.0,177562.0,185686.0,234573.0,8424.0,10663.0,11548.0,12555.0,13332.0,16113.0,93202.0,65.927241,67.0,97.895408,7.0,99.808673,81746.0,53.489239,65.0,97.959834,3.0,99.936245,70429.0,42.906838,55.0,98.279159,4.0,99.904398,57011.0,32.107658,52.0,98.376313,6.0,99.840815,48887.0,26.327779,49.0,98.471824,10.0,99.713467
618,"Comal County, Texas",091,48,104960.0,106111.0,112083.0,119632.0,124234.0,156257.0,5223.0,5389.0,5810.0,6274.0,6540.0,8575.0,51297.0,48.872904,151.0,95.216837,20.0,99.394133,50146.0,47.258060,131.0,95.855913,6.0,99.840612,44174.0,39.411864,119.0,96.239643,5.0,99.872530,36625.0,30.614718,101.0,96.816301,8.0,99.777141,32023.0,25.776358,97.0,96.943649,12.0,99.649793
1476,"Kaufman County, Texas",257,48,95993.0,101197.0,105220.0,109289.0,111830.0,140145.0,6167.0,6196.0,6375.0,6595.0,6738.0,8800.0,44152.0,45.995020,181.0,94.260204,26.0,99.202806,38948.0,38.487307,167.0,94.708320,16.0,99.521836,34925.0,33.192359,150.0,95.251753,18.0,99.458254,30856.0,28.233400,131.0,95.861191,12.0,99.649793,28315.0,25.319682,118.0,96.275072,13.0,99.617956
2161,"Osceola County, Florida",097,12,254739.0,265328.0,279837.0,300870.0,311962.0,380331.0,17299.0,17647.0,18778.0,20802.0,21933.0,26521.0,125592.0,49.302227,47.0,98.533163,17.0,99.489796,115003.0,43.343710,44.0,98.629264,12.0,99.649347,100494.0,35.911620,41.0,98.725303,9.0,99.745061,79461.0,26.410410,33.0,98.981216,15.0,99.554282,68369.0,21.915810,32.0,99.013053,19.0,99.426934
2666,"St. Johns County, Florida",109,12,174994.0,186077.0,197115.0,210495.0,218362.0,265724.0,9314.0,8752.0,9353.0,9994.0,10689.0,11314.0,90730.0,51.847492,68.0,97.863520,15.0,99.553571,79647.0,42.803248,69.0,97.832324,13.0,99.617469,68609.0,34.806585,58.0,98.183556,14.0,99.585723,55229.0,26.237678,53.0,98.344476,16.0,99.522445,47362.0,21.689671,50.0,98.439987,20.0,99.395097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,"Baltimore city, Maryland",510,24,639337.0,620210.0,621445.0,622454.0,621000.0,592211.0,59583.0,56849.0,59009.0,61135.0,61842.0,56338.0,-47126.0,-7.371073,3134.0,0.095663,2777.0,11.479592,-27999.0,-4.514439,3136.0,0.063755,2315.0,26.235257,-29234.0,-4.704197,3138.0,0.031867,2442.0,22.211600,-30243.0,-4.858672,3141.0,0.031837,2581.0,17.860554,-28789.0,-4.635910,3140.0,0.063674,2624.0,16.491563
339,"Caddo Parish, Louisiana",017,22,252467.0,253924.0,255551.0,254742.0,253125.0,239775.0,19563.0,18751.0,18864.0,18482.0,18469.0,16421.0,-12692.0,-5.027192,3126.0,0.350765,2533.0,19.260204,-14149.0,-5.572140,3130.0,0.255021,2508.0,20.082882,-15776.0,-6.173327,3136.0,0.095602,2659.0,15.296367,-14967.0,-5.875356,3135.0,0.222859,2715.0,13.594397,-13350.0,-5.274074,3136.0,0.191022,2709.0,13.785419
3006,"Wayne County, North Carolina",191,37,113290.0,121324.0,123382.0,124355.0,124447.0,

In [25]:
df_1m_pop_pct_changes = df_growth_data_by_year_pivot.query(
    f"Total_Pop_{range_start_year} >= 1000000").sort_values(sort_column).dropna(
    subset = sort_column).copy()
pd.concat([df_1m_pop_pct_changes.head(), df_1m_pop_pct_changes.tail()])[
['NAME', f'{acs5_year - range_for_sorting}-{acs5_year} \
Total_Pop Change', f'{
acs5_year - range_for_sorting}-{acs5_year} \
Total_Pop % Change',
f'{acs5_year - range_for_sorting}-\
{acs5_year} Change Rank', f'{acs5_year - range_for_sorting}\
-{acs5_year} Change Percentile']]

,NAME,2016-2021 Total_Pop Change,2016-2021 Total_Pop % Change,2016-2021 Change Rank,2016-2021 Change Percentile
2145,"Orange County, Florida",153894.0,12.252170,5.0,99.872652
2824,"Travis County, Texas",119619.0,10.418176,12.0,99.649793
1273,"Hillsborough County, Florida",121300.0,9.168147,11.0,99.681630
1883,"Mecklenburg County, North Carolina",89210.0,8.817186,18.0,99.458771
529,"Clark County, Nevada",160994.0,7.776913,4.0,99.904489
1915,"Miami-Dade County, Florida",25695.0,0.964376,130.0,95.893028
630,"Cook County, Illinois",37823.0,0.723529,71.0,97.771410
703,"Cuyahoga County, Ohio",4957.0,0.393816,446.0,85.832537
2672,"St. Louis County, Missouri",1422.0,0.142120,773.0,75.421840
1718,"Los Angeles County, California",-37520.0,-0.373068,3141.0,0.031837


Saving this data as a .csv file so that we can create maps of it within the Mapping section of Python for Nonprofits:

In [26]:
df_growth_data_by_year_pivot.to_csv(
    f'Datasets/grad_destinations_acs_data.csv', 
    index = False)

# Appendix

## 1: The Census Python Library

It's worth noting that there is also a 'census' Python library (available via pypi and conda) that helps simplify the process of requesting API data. You may choose to use it for your own Census research, but I ended up not needing it for the data retrieval tasks shown above. In addition, foregoing the library allowed me to demonstrate how to retrieve data directly from an API, which you may find helpful when working with APIs that don't have a corresponding Python library. 

Here's an example of the Census library in use:

In [27]:
## Example of reading data from the Census library into a 
# Pandas DataFrame:
from census import Census
c = Census(key)
pd.DataFrame(c.acs5.get(('NAME', 'B01001_001E'),
{'for': 'county:*'}))

,NAME,B01001_001E,state,county
0,"Autauga County, Alabama",58761.0,01,001
1,"Baldwin County, Alabama",233420.0,01,003
2,"Barbour County, Alabama",24877.0,01,005
3,"Bibb County, Alabama",22251.0,01,007
4,"Blount County, Alabama",59077.0,01,009
...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",54182.0,72,145
3218,"Vieques Municipio, Puerto Rico",8199.0,72,147
3219,"Villalba Municipio, Puerto Rico",21984.0,72,149
3220,"Yabucoa Municipio, Puerto Rico",30313.0,72,151


## 2: The requests library

We can also use Python's *requests* library to retrieve data from the Census API, then convert it to JSON format:

In [28]:
# The following code borrows from the requests library documentation at 
# https://docs.python-requests.org/en/latest/index.html
import requests
r = requests.get(f'https://api.census.gov/data/{acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# Printing the first 300 characters of this output:
print("r.text:\n",r.text[0:300],'\n')
# Printing the first 5 lines of r.json:
print("r.json:\n",r.json()[0:5],'\n')

r.text:
 [["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58239","01","001"],
["Baldwin County, Alabama","227131","01","003"],
["Barbour County, Alabama","25259","01","005"],
["Bibb County, Alabama","22412","01","007"],
["Blount County, Alabama","58884","01","009"],
["Bullock County, Ala 

r.json:
 [['NAME', 'B01001_001E', 'state', 'county'], ['Autauga County, Alabama', '58239', '01', '001'], ['Baldwin County, Alabama', '227131', '01', '003'], ['Barbour County, Alabama', '25259', '01', '005'], ['Bibb County, Alabama', '22412', '01', '007']] 



Converting our response to JSON allows it to be easily read into a Pandas DataFrame, as shown below:

In [29]:
pd.DataFrame(r.json()).head()
# Note that pd.DataFrame(r.text) would produce the following error:
# "ValueError: DataFrame constructor not properly called!"

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58239,01,001
2,"Baldwin County, Alabama",227131,01,003
3,"Barbour County, Alabama",25259,01,005
4,"Bibb County, Alabama",22412,01,007


I included this approach in the appendix because you may find the requests library useful for other online data retrieval tasks. However, our use of `pd.read_json()` to import Census data rendered an explicit call to the requests library unnecessary.

In [30]:
program_end_time = time.time()
run_time = round(program_end_time - program_start_time, 3)
print(f"Finished running script in {run_time} seconds.")

Finished running script in 17.459 seconds.
